# Lectura de paquetes y data

In [1]:
import warnings
import os
import time
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from statsmodels.tsa.seasonal import STL

from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna

# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils
from src.utils_ml import ml_plotting as plot_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniconda/base/envs/maestria/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Utilizamos paths relativos para la lectura de la data

In [2]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Current absolute path: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/notebooks

BASE_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi
DATA_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data
OUTPUT_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data/output


In [3]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

(6834, 11)

In [4]:
df_base.head(5)

,Fe.prefer.entrega,Day_of_the_Week,Pedidos,Sell_In,Sell_in_on_time,CEDIS,SKU,Mes_Año,Sell_in_rezago,porcentaje_on_time,Subcategoria
0,2024-07-05,Friday,16200,16200,16200,7,SKU7,72024,0,1.0,BEBIDAS DE YOGUR
1,2024-07-05,Friday,19800,19800,19800,6,SKU5,72024,0,1.0,BEBIDAS DE YOGUR
2,2024-07-05,Friday,4950,4950,4950,4,SKU6,72024,0,1.0,BEBIDAS DE YOGUR
3,2024-07-05,Friday,8100,8100,8100,6,SKU4,72024,0,1.0,BEBIDAS DE YOGUR
4,2024-07-05,Friday,18600,18600,18600,7,SKU3,72024,0,1.0,BEBIDAS DE YOGUR


In [5]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [6]:
df

,Fecha,SKU,Pedidos
0,2024-07-05,SKU7,16200
1,2024-07-05,SKU5,19800
2,2024-07-05,SKU6,4950
3,2024-07-05,SKU4,8100
4,2024-07-05,SKU3,18600
...,...,...,...
6829,2025-04-10,SKU14,1560
6830,2025-04-10,SKU16,3900
6831,2025-04-10,SKU8,4200
6832,2025-04-10,SKU22,2160


# Preparación de la data

In [7]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [8]:
# Modificar nombre de columnas
df.columns = df.columns.str.replace(".", "_", regex=False).str.lower()

In [9]:
df.shape

(7280, 3)

# EDA

## general

In [10]:
df.isna().sum()

fecha      0
sku        0
pedidos    0
dtype: int64

In [11]:
# porcentaje de ceros por sku
porcentaje_ceros = (
    df.groupby("sku")["pedidos"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index(name="prct_ceros")
    .round(2)
)

# promedio, mediana y desviacion estandar por sku excluyendo ceros
df_temp = df[df["pedidos"] > 0].copy()
promedio = df_temp.groupby("sku")["pedidos"].mean().reset_index(name="Promedio").round()
mediana = df_temp.groupby("sku")["pedidos"].median().reset_index(name="Mediana").round()
desviacion = (
    df_temp.groupby("sku")["pedidos"].std().reset_index(name="Desviacion").round()
)
maximo = df_temp.groupby("sku")["pedidos"].max().reset_index(name="Maximo").round()

# Porcentaje de valores outliers por SKU excluyendo ceros
porcentaje_outliers = (
    df_temp.groupby("sku")["pedidos"]
    .apply(fe_utils.calcular_outliers_porcentaje)
    .reset_index(name="prct_outliers")
    .round(2)
)

# Unir las tablas
tabla_total = pd.merge(porcentaje_ceros, porcentaje_outliers, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, promedio, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, mediana, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, desviacion, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, maximo, on="sku", how="outer")
tabla_total.sort_values(by="prct_ceros", ascending=False)

,sku,prct_ceros,prct_outliers,Promedio,Mediana,Desviacion,Maximo
6,SKU15,16.79,7.30,1004.0,840.0,647.0,3960
7,SKU16,14.29,2.08,4624.0,4688.0,1172.0,7500
0,SKU1,12.50,1.63,17878.0,15000.0,13859.0,84600
24,SKU8,11.79,5.26,1067.0,900.0,893.0,5160
4,SKU13,11.07,4.42,502.0,420.0,285.0,1500
3,SKU12,10.36,3.59,460.0,420.0,283.0,1800
8,SKU17,7.86,2.71,11496.0,9030.0,7494.0,52200
25,SKU9,7.86,8.53,799.0,600.0,641.0,6000
5,SKU14,7.86,2.71,2706.0,1920.0,2088.0,15240
10,SKU19,6.07,3.42,3103.0,2640.0,1959.0,10800


## Tendencias

In [12]:
sku = "SKU5"
print(f"Analizando el SKU: {sku}")

Analizando el SKU: SKU5


In [13]:
df_sku = df[df["sku"] == sku].copy()
df_sku = df_sku.drop("sku", axis=1)

# graficamos la serie de tiempo del SKU seleccionado usando plotly
fig = px.line(
    df_sku,
    x="fecha",
    y="pedidos",
    title=f"Serie de tiempo de Pedidos para {sku}:",
)
fig.update_layout(
    xaxis_title="Fecha",
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_traces(line=dict(color="blue", width=2))
fig.show()


### Descomposición STL

In [14]:
# graficamos la tendencia y estacionalidad de cada SKU usando STL
plot_utils.graficar_serie_con_descomposicion(df_sku, sku=sku, periodo=7)


# Feature engineering

## Variables temporales

In [15]:
df = fe_utils.create_temporal_features(df, "fecha")
df.shape, df.columns

((7280, 10),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena',
        'is_vacation'],
       dtype='object'))

## Variables tipo lag

In [16]:
df = fe_utils.create_lag_features(
    df, "pedidos", "sku", "fecha", max_daily_lag=14, weekday_lags=3
)
df.shape, df.columns

((7280, 27),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3'],
       dtype='object'))

## Variables tipo promedio moviles

In [17]:
df = fe_utils.create_rolling_features(df, "pedidos", "sku", "fecha")
df.shape, df.columns

((7280, 32),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3', 'pedidos_rolling_2',
        'pedidos_rolling_7', 'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks',
        'pedidos_dow_avg_3wks'],
       dtype='object'))

## Variables lags de STL

In [18]:
df = fe_utils.create_stl_features(
    df, "pedidos", "sku", "fecha", seasonal=7, stl_lags=14
)
df.shape, df.columns


((7280, 60),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3', 'pedidos_rolling_2',
        'pedidos_rolling_7', 'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks',
        'pedidos_dow_avg_3wks', 'stl_trend_lag_1', 'stl_trend_lag_2',
        'stl_trend_lag_3', 'stl_trend_lag_4', 'stl_trend_lag_5',
        'stl_trend_lag_6', 'stl_trend_lag_7', 'stl_trend_lag_8',
        'stl_trend_lag_9', 'stl_trend_lag_10', 'stl_trend_lag_11',
        'stl_trend_lag_12', 'stl_trend_lag_13', 'stl_trend_lag_14',
        'stl_seasonal_lag_1', 'stl

## Aplanamiento de outliers en demanda

In [19]:
df = fe_utils.cap_upper_outliers(df, "pedidos", "sku")

## Ajustes finales a la data

In [20]:
# Eliminamos todas las filas con valores NaN para que no afecten el entrenamiento
df.dropna(inplace=True)
df.shape, df.columns

((6543, 60),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3', 'pedidos_rolling_2',
        'pedidos_rolling_7', 'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks',
        'pedidos_dow_avg_3wks', 'stl_trend_lag_1', 'stl_trend_lag_2',
        'stl_trend_lag_3', 'stl_trend_lag_4', 'stl_trend_lag_5',
        'stl_trend_lag_6', 'stl_trend_lag_7', 'stl_trend_lag_8',
        'stl_trend_lag_9', 'stl_trend_lag_10', 'stl_trend_lag_11',
        'stl_trend_lag_12', 'stl_trend_lag_13', 'stl_trend_lag_14',
        'stl_seasonal_lag_1', 'stl

In [21]:
df

,fecha,sku,pedidos,day_of_week,day_of_month,is_weekend,week_of_month,is_start_of_month,is_near_quincena,is_vacation,pedidos_lag_1,pedidos_lag_2,pedidos_lag_3,pedidos_lag_4,pedidos_lag_5,pedidos_lag_6,pedidos_lag_7,pedidos_lag_8,pedidos_lag_9,pedidos_lag_10,pedidos_lag_11,pedidos_lag_12,pedidos_lag_13,pedidos_lag_14,pedidos_weekday_lag_1,pedidos_weekday_lag_2,pedidos_weekday_lag_3,pedidos_rolling_2,pedidos_rolling_7,pedidos_prev_week_avg,pedidos_dow_avg_2wks,pedidos_dow_avg_3wks,stl_trend_lag_1,stl_trend_lag_2,stl_trend_lag_3,stl_trend_lag_4,stl_trend_lag_5,stl_trend_lag_6,stl_trend_lag_7,stl_trend_lag_8,stl_trend_lag_9,stl_trend_lag_10,stl_trend_lag_11,stl_trend_lag_12,stl_trend_lag_13,stl_trend_lag_14,stl_seasonal_lag_1,stl_seasonal_lag_2,stl_seasonal_lag_3,stl_seasonal_lag_4,stl_seasonal_lag_5,stl_seasonal_lag_6,stl_seasonal_lag_7,stl_seasonal_lag_8,stl_seasonal_lag_9,stl_seasonal_lag_10,stl_seasonal_lag_11,stl_seasonal_lag_12,stl_seasonal_lag_13,stl_seasonal_lag_14
21,2024-07-26,SKU1,0.0,4,26,0,4,0,0,1,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,2400.0,2400.0,0.0,0.0,6000.0,0.0,7200.0,42000.0,15857.142857,13200.000000,6000.0,6600.0,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,11318.073301,11162.328801,11142.505288,11307.379638,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590,-8702.020679,-7241.913414,-9376.524091,-9032.297251
22,2024-07-27,SKU1,0.0,5,27,1,4,0,0,1,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,2400.0,2400.0,0.0,3000.0,0.0,7200.0,30000.0,17500.000000,11742.857143,3000.0,5100.0,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,11318.073301,11162.328801,11142.505288,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590,-8702.020679,-7241.913414,-9376.524091
24,2024-07-29,SKU1,6000.0,0,29,0,5,0,1,1,6000.0,0.0,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,2400.0,6000.0,2400.0,3000.0,6000.0,20400.000000,12771.428571,4200.0,3800.0,11759.561758,11686.683023,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,11318.073301,-6275.247195,-10356.559802,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590,-8702.020679
25,2024-07-30,SKU1,6000.0,1,30,0,5,0,1,1,6000.0,6000.0,0.0,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,6000.0,2400.0,3000.0,6000.0,20400.000000,13285.714286,4200.0,3800.0,12147.310782,11759.561758,11686.683023,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,-6213.341647,-6275.247195,-10356.559802,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590
26,2024-07-31,SKU1,36000.0,2,31,0,5,0,1,1,6000.0,6000.0,6000.0,0.0,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,54000.0,36000.0,24000.0,6000.0,20400.000000,15857.142857,45000.0,38000.0,12733.007654,12147.310782,11759.561758,11686.683023,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,-6884.697557,-6213.341647,-6275.247195,-10356.559802,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

# Modelling 

In [22]:
## Preparación de data para modelling

# Separamos el conjunto de datos en entrenamiento y prueba, usando los últimos 7 días como prueba
test = df.tail(7)
df_mod = df[:-7].copy()

# Identificamos las variables predictoras
features = df_mod.columns.difference(["fecha", "sku", "pedidos"]).to_list()

# definimos el numero de ventanas de evaluación y el tamaño de las ventanas
val_iter = 3
val_size = 7

# definimos el tamaño inicial o base del conjunto de entrenamiento
train_size = df_mod.shape[0] - val_iter * val_size

## Evaluación modelos XGBoost

In [23]:
df_sku = df_mod[df_mod["sku"] == sku].copy()

# definimos el tamaño inicial o base del conjunto de entrenamiento
train_size = df_sku.shape[0] - val_iter * val_size


model = XGBRegressor(max_depth=2, n_estimators=100, learning_rate=0.1, random_state=42)
avg_rmse, avg_gap = ml_utils.evaluate_model_with_recursive_window(
    model,
    X=df_sku[features],
    y=df_sku["pedidos"],
    val_iter=val_iter,
    start_train=0,
    train_size=train_size,
    val_size=val_size,
)


RMSE Entrenamiento: 2798.6919, RMSE Validación: 6622.4517, GAP: 3823.7598
RMSE Entrenamiento: 2910.5008, RMSE Validación: 1435.0909, GAP: 1475.4099
RMSE Entrenamiento: 2925.4004, RMSE Validación: 1521.7017, GAP: 1403.6988


In [ ]:
# Filtramos la data por SKU
df_sku = df_mod[df_mod["sku"] == sku].copy()

# definimos el tamaño inicial o base del conjunto de entrenamiento
train_size = df_sku.shape[0] - val_iter * val_size

param_grid = {
    "max_depth": lambda trial: trial.suggest_int("max_depth", 2, 8, step=1),
    "learning_rate": lambda trial: trial.suggest_float(
        "learning_rate", 0.001, 0.3, step=0.001
    ),
    "n_estimators": lambda trial: trial.suggest_int(
        "n_estimators", 100, 3000, step=100
    ),
    "subsample": lambda trial: trial.suggest_float("subsample", 0.6, 1.0, step=0.02),
    "colsample_bytree": lambda trial: trial.suggest_float(
        "colsample_bytree", 0.6, 1.0, step=0.02
    ),
    "gamma": lambda trial: trial.suggest_float("gamma", 0.0, 8.0, step=0.1),
    "reg_alpha": lambda trial: trial.suggest_float("reg_alpha", 0.0, 8.0, step=0.1),
    "reg_lambda": lambda trial: trial.suggest_float("reg_lambda", 0.0, 8.0, step=0.1),
    "min_child_weight": lambda trial: trial.suggest_int(
        "min_child_weight", 3, 15, step=1
    ),
}

study = ml_utils.optimize_model_with_optuna(
    model_class=XGBRegressor,
    param_grid=param_grid,
    X=df_sku[features],
    y=df_sku["pedidos"],
    n_trials=10,
    val_iter=val_iter,
    train_size=train_size,
    val_size=val_size,
    show_progress_bar=True,
)

## Evaluación modelos Random Forest

## Evaluación modelos Elastic Net

## Selección mejor modelo y ajuste final

# Predicción y graficas 